In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel
import ast

In [ ]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/ExternData.csv'
train_full = pd.read_csv(archivo_3)

# AST spliter D. Gries form

In [ ]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [ ]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function

def DGries_states(solution):
  try:
    # Unescape newline characters
    solution = solution.replace('\\n', '\n')
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    raise error
    #return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)

In [ ]:
train_full.dropna(inplace=True)

In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Python Code'].apply(DGries_states).apply(pd.Series)
display(train_full.head())

OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body


,Problem Description,Python Code,Error Label,Source,Estado incial,Transformación de estado,Estado final
0,Count how many even numbers are up to n. Logic...,def count_evens_buggy(n):\n i = 0\n coun...,2,Inspired by CodeChef Learn Python (While Loop)...,i = 0\n count = 0,if i % 2 == 0:\n count += 1,i <= n
1,Sum numbers from 1 to n. Error: i is incorrect...,def sum_to_n_buggy(n):\n total = 0\n i =...,2,Based on typical mistakes in Codeforces common...,total = 0\n i = 1,total += i\ni = i + 0,i <= n
2,Prompt user for numbers until 0 is entered. Er...,def input_until_zero_buggy():\n num = int(i...,2,Common error from GeeksforGeeks Python while l...,num = int(input('Enter number (0 to exit): ')),"print('Number:', num)",num != 0
3,Find first multiple of 7 greater than n. Error...,def first_multiple_7_buggy(n):\n i = n + 1\...,2,Typical error in CodeNet beginner loop problems,i = n + 1,pass,i % 7 != 0
4,Calculate factorial of n. Logical error: decre...,def factorial_buggy(n):\n result = 1\n w...,2,Inspired by classic factorial exercises,result = 1,result *= n\nn -= 2,n > 1


In [ ]:
# Assuming the error occurs on the first row, inspect the 'Python Code' column
print(train_full['Python Code'].iloc[0])

def count_evens_buggy(n):\n    i = 0\n    count = 0\n    while i <= n:\n        if i % 2 == 0:\n            count += 1\n        # missing i += 1\n    return count


# Load Dataset

In [ ]:
train_full

,Problem Description,Python Code,Error Label,Source,Estado incial,Transformación de estado,Estado final
0,Count how many even numbers are up to n. Logic...,def count_evens_buggy(n):\n i = 0\n coun...,2,Inspired by CodeChef Learn Python (While Loop)...,i = 0\n count = 0,if i % 2 == 0:\n count += 1,i <= n
1,Sum numbers from 1 to n. Error: i is incorrect...,def sum_to_n_buggy(n):\n total = 0\n i =...,2,Based on typical mistakes in Codeforces common...,total = 0\n i = 1,total += i\ni = i + 0,i <= n
2,Prompt user for numbers until 0 is entered. Er...,def input_until_zero_buggy():\n num = int(i...,2,Common error from GeeksforGeeks Python while l...,num = int(input('Enter number (0 to exit): ')),"print('Number:', num)",num != 0
3,Find first multiple of 7 greater than n. Error...,def first_multiple_7_buggy(n):\n i = n + 1\...,2,Typical error in CodeNet beginner loop problems,i = n + 1,pass,i % 7 != 0
4,Calculate factorial of n. Logical error: decre...,def factorial_buggy(n):\n result = 1\n w...,2,Inspired by classic factorial exercises,result = 1,result *= n\nn -= 2,n > 1
5,Count digits in a positive integer n. Error: n...,def count_digits_buggy(n):\n count = 0\n ...,2,Inspired by digit counting problems in Codefor...,count = 0,n % 10\ncount += 1,n > 0
6,Sum all even numbers up to n. Error: loop cond...,def sum_evens_buggy(n):\n total = 0\n i ...,1,"CodeChef/Codeforces summing problems, conditio...",total = 0\n i = 2,total += i\ni += 2,i < n
7,Find a prime number using break. Error: break ...,def find_prime_buggy():\n num = 2\n whil...,2,Inspired by break/continue loop errors from Py...,num = 2,is_prime = True\ndivisor = 2\nwhile divisor < ...,True
8,Print cubes of numbers from 1 to n. Error: i n...,def print_cubes_buggy(n):\n while i <= n:\n...,0,"Common initialization error, based on CodeNet ...",print(i ** 3)\ni += 1,print(i ** 3)\ni += 1,i <= n
9,Prompt for password until correct. Error: pass...,def prompt_password_buggy():\n password = i...,2,Typical validation loop error from Python begi...,password = input('Enter password: '),print('Incorrect'),password != 'secret'


In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Python Code'].apply(DGries_states).apply(pd.Series)

OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body


In [ ]:
train_full.dropna(inplace=True)

In [ ]:
np.unique(train_full['Error Label'])

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
train_full.head()

,Problem Description,Python Code,Error Label,Source,Estado incial,Transformación de estado,Estado final
0,Count how many even numbers are up to n. Logic...,def count_evens_buggy(n):\n i = 0\n coun...,2,Inspired by CodeChef Learn Python (While Loop)...,i = 0\n count = 0,if i % 2 == 0:\n count += 1,i <= n
1,Sum numbers from 1 to n. Error: i is incorrect...,def sum_to_n_buggy(n):\n total = 0\n i =...,2,Based on typical mistakes in Codeforces common...,total = 0\n i = 1,total += i\ni = i + 0,i <= n
2,Prompt user for numbers until 0 is entered. Er...,def input_until_zero_buggy():\n num = int(i...,2,Common error from GeeksforGeeks Python while l...,num = int(input('Enter number (0 to exit): ')),"print('Number:', num)",num != 0
3,Find first multiple of 7 greater than n. Error...,def first_multiple_7_buggy(n):\n i = n + 1\...,2,Typical error in CodeNet beginner loop problems,i = n + 1,pass,i % 7 != 0
4,Calculate factorial of n. Logical error: decre...,def factorial_buggy(n):\n result = 1\n w...,2,Inspired by classic factorial exercises,result = 1,result *= n\nn -= 2,n > 1


In [ ]:
encoder=EncodeTextSource()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
%%time
problem=train_full['Problem Description'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Python Code'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_codes).to_numpy()

CPU times: user 3.9 s, sys: 1.73 s, total: 5.64 s
Wall time: 8.62 s


In [ ]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((35,), (35,), (35,), (35,), (35,))

In [ ]:
print(startstate.shape)
print(startstate[30].shape)


(35,)
(1, 10, 768)


In [ ]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((35, 768), (35, 768), (35, 768), (35, 768))

In [ ]:
y=train_full['Error Label']
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6])

In [ ]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [ ]:
from tensorflow import keras
from time import time
accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]
models_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]


Model_Xtest=[Xp,Xcode,Xs,Xt,Xf]
for i,my_model in enumerate(models_names):
  model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/ablations_finall_{0}.keras'.format(my_model))
  if(i==1):
    Model_Xtest=[Xp,Xs,Xt,Xf]
  if(i==2):
    Model_Xtest=[Xcode,Xs,Xt,Xf]
  if(i==3):
    Model_Xtest=[Xs,Xt,Xf]
  if(i==4):
    Model_Xtest=[Xp,Xcode]
  if(i==5):
    Model_Xtest=[Xcode]


  evaluate=model.evaluate(Model_Xtest,y)

  t1=time()
  y_pred=model.predict(Model_Xtest)
  t2=time()
  mcc=calculate_mcc_multiclass(y, y_pred)
  auc_pr=calculate_auc_pr_multiclass(y, y_pred)

  times_predict.append(t2-t1)
  accuracies_predict.append(evaluate[1])
  loss_predict.append(evaluate[0])
  mccs_predict.append(mcc)
  aucpr_predict.append(auc_pr)


  accuracy=evaluate[1]
  loss=evaluate[0]

  print("accuracy",accuracy)
  print("loss",loss)
  print("mcc",mcc)
  print("auc_pr",auc_pr)
  print("time predict",t2-t1)

2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.2946 - loss: 2.8522
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step
accuracy 0.2857142984867096
loss 2.9015676975250244
mcc 0.06268035106150013
auc_pr 0.2069570819281064
time predict 4.781700372695923


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 875ms/step - accuracy: 0.2652 - loss: 2.7634
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step   
accuracy 0.2571428716182709
loss 2.7938945293426514
mcc 0.015166081788042879
auc_pr 0.20332598676705735
time predict 2.5224905014038086


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 514ms/step - accuracy: 0.2946 - loss: 2.5657


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 709ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 725ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


accuracy 0.2857142984867096
loss 2.6152827739715576
mcc 0.09691098082930297
auc_pr 0.2196274521897154
time predict 1.4772028923034668


In [ ]:
model=keras.models.load_model('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/ablations_finall_{0}.keras'.format("problemall"))


In [ ]:
Model_Xtest=[Xp,Xcode,Xs,Xt,Xf]
y_pred=model.predict(Model_Xtest)

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step


In [ ]:
train_full["y_pred"]=y_pred.argmax(axis=1)

In [ ]:
train_full.to_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/test_best_modelbert.csv')

In [ ]:
import pandas as pd
df_final=pd.read_csv('/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/test_best_modelbert.csv')

In [ ]:
df_final

,Unnamed: 0,Problem Description,Python Code,Error Label,Source,Estado incial,Transformación de estado,Estado final,y_pred
0,0,Count how many even numbers are up to n. Logic...,def count_evens_buggy(n):\n i = 0\n coun...,2,Inspired by CodeChef Learn Python (While Loop)...,i = 0\n count = 0,if i % 2 == 0:\n count += 1,i <= n,0
1,1,Sum numbers from 1 to n. Error: i is incorrect...,def sum_to_n_buggy(n):\n total = 0\n i =...,2,Based on typical mistakes in Codeforces common...,total = 0\n i = 1,total += i\ni = i + 0,i <= n,5
2,2,Prompt user for numbers until 0 is entered. Er...,def input_until_zero_buggy():\n num = int(i...,2,Common error from GeeksforGeeks Python while l...,num = int(input('Enter number (0 to exit): ')),"print('Number:', num)",num != 0,0
3,3,Find first multiple of 7 greater than n. Error...,def first_multiple_7_buggy(n):\n i = n + 1\...,2,Typical error in CodeNet beginner loop problems,i = n + 1,pass,i % 7 != 0,2
4,4,Calculate factorial of n. Logical error: decre...,def factorial_buggy(n):\n result = 1\n w...,2,Inspired by classic factorial exercises,result = 1,result *= n\nn -= 2,n > 1,2
5,5,Count digits in a positive integer n. Error: n...,def count_digits_buggy(n):\n count = 0\n ...,2,Inspired by digit counting problems in Codefor...,count = 0,n % 10\ncount += 1,n > 0,2
6,6,Sum all even numbers up to n. Error: loop cond...,def sum_evens_buggy(n):\n total = 0\n i ...,1,"CodeChef/Codeforces summing problems, conditio...",total = 0\n i = 2,total += i\ni += 2,i < n,0
7,7,Find a prime number using break. Error: break ...,def find_prime_buggy():\n num = 2\n whil...,2,Inspired by break/continue loop errors from Py...,num = 2,is_prime = True\ndivisor = 2\nwhile divisor < ...,True,0
8,8,Print cubes of numbers from 1 to n. Error: i n...,def print_cubes_buggy(n):\n while i <= n:\n...,0,"Common initialization error, based on CodeNet ...",print(i ** 3)\ni += 1,print(i ** 3)\ni += 1,i <= n,0
9,9,Prompt for password until correct. Error: pass...,def prompt_password_buggy():\n password = i...,2,Typical validation loop error from Python begi...,password = input('Enter password: '),print('Incorrect'),password != 'secret',0


In [ ]:
model.fi

Epoch 1/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 23s 23s/step - accuracy: 0.8857 - loss: 0.6587 - learning_rate: 0.0010
Epoch 2/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.8857 - loss: 0.5464 - learning_rate: 0.0010
Epoch 3/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.8857 - loss: 0.3989 - learning_rate: 0.0010
Epoch 4/1000


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)
/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/callback_list.py:145: UserWarning: Learning rate reduction is conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss,learning_rate.
  callback.on_epoch_end(epoch, logs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.9143 - loss: 0.3124 - learning_rate: 0.0010
Epoch 5/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.9714 - loss: 0.2243 - learning_rate: 0.0010
Epoch 6/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.9714 - loss: 0.1787 - learning_rate: 0.0010
Epoch 7/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 1.0000 - loss: 0.1025 - learning_rate: 0.0010
Epoch 8/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - accuracy: 1.0000 - loss: 0.1165 - learning_rate: 0.0010
Epoch 9/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 1.0000 - loss: 0.0976 - learning_rate: 0.0010
Epoch 10/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 1.0000 - loss: 0.0959 - learning_rate: 0.0010
Epoch 11/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 1.0000 - loss: 0.0850 - learning_rate: 0.0010
Epoch 12/1000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 1.0000 - loss: 0.0765 - learning_rate: 0.0010
Epoch 13/1000
1/1 ━━━━━━━━━